# 1. Setup
Run every cell in this section at the start of each Kaggle session (GPU on, internet on).

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = "/root/.local/bin:/usr/local/bin:" + os.environ["PATH"]
os.environ["UV_PROJECT_ENVIRONMENT"] = "/tmp/dqn-venv"
!uv --version

In [ ]:
import os
if not os.path.exists("/kaggle/working/Deep-Q-Learning"):
    !git clone https://github.com/Ebi-Tech/Deep-Q-Learning.git /kaggle/working/Deep-Q-Learning
%cd /kaggle/working/Deep-Q-Learning

In [ ]:
!uv sync

In [ ]:
%%bash
uv run python - <<'EOF'
import os
import sys

import torch

failures = 0
def check(name, condition):
    global failures
    print("ok    " if condition else "FAILED", name)
    if not condition:
        failures += 1

check("train.py exists", os.path.isfile("train.py"))
check(f"current folder is the repo root: {os.getcwd()}",
      os.getcwd() == "/kaggle/working/Deep-Q-Learning" and os.path.isfile("pyproject.toml"))
check(f"CUDA is available: {torch.cuda.is_available()}", torch.cuda.is_available())

import train
sys.argv = ["train.py", "--member", "Divine", "--experiment", "0"]
train_every = train.get_settings().train_every
check(f"default train_every in train.py is 1: {train_every}", train_every == 1)

print("all checks passed" if failures == 0 else f"{failures} check(s) FAILED")
EOF

In [ ]:
import csv
import os

def check_experiment(n):
    """Check that experiment n finished, then print its average reward over the last 100 episodes."""
    log_path = f"logs/Divine/experiment_{n}.csv"
    model_path = f"models/Divine/experiment_{n}.pth"
    rows = []
    if os.path.isfile(log_path):
        with open(log_path) as f:
            rows = list(csv.DictReader(f))
    last_step = int(rows[-1]["step"]) if rows else 0
    checks = [
        (f"log {log_path} exists", os.path.isfile(log_path)),
        (f"last step is at least 999,000: {last_step:,}", last_step >= 999_000),
        (f"model {model_path} exists", os.path.isfile(model_path)),
    ]
    for name, passed in checks:
        print("ok    " if passed else "FAILED", name)
    if rows:
        last_100 = [float(row["reward"]) for row in rows[-100:]]
        print(f"average reward over the last {len(last_100)} episodes: {sum(last_100) / len(last_100):.2f}")
    print("all checks passed" if all(passed for _, passed in checks) else "some checks FAILED")

# 2. Experiments
One cell per experiment, so they can be split across Kaggle sessions. Each run uses the defaults in train.py except the setting named above it.

Exp 1: baseline, all default settings.

In [ ]:
!uv run train.py --member Divine --experiment 1
check_experiment(1)

Exp 2: learning rate 5e-5 (lower than baseline).

In [ ]:
!uv run train.py --member Divine --experiment 2 --learning_rate 5e-5
check_experiment(2)

Exp 3: learning rate 2.5e-4 (higher than baseline).

In [ ]:
!uv run train.py --member Divine --experiment 3 --learning_rate 2.5e-4
check_experiment(3)

Exp 4: learning rate 5e-4.

In [ ]:
!uv run train.py --member Divine --experiment 4 --learning_rate 5e-4
check_experiment(4)

Exp 5: learning rate 1e-3.

In [ ]:
!uv run train.py --member Divine --experiment 5 --learning_rate 1e-3
check_experiment(5)

Exp 6: gamma 0.95 (less weight on future rewards).

In [ ]:
!uv run train.py --member Divine --experiment 6 --gamma 0.95
check_experiment(6)

Exp 7: gamma 0.9.

In [ ]:
!uv run train.py --member Divine --experiment 7 --gamma 0.9
check_experiment(7)

Exp 8: gamma 0.999 (more weight on future rewards).

In [ ]:
!uv run train.py --member Divine --experiment 8 --gamma 0.999
check_experiment(8)

Exp 9: best learning rate + best gamma. Exp 10: second best learning rate + best gamma. Fill these in after experiments 2 to 8.

In [ ]:
# !uv run train.py --member Divine --experiment 9 --learning_rate BEST_LR --gamma BEST_GAMMA
# check_experiment(9)

In [ ]:
# !uv run train.py --member Divine --experiment 10 --learning_rate SECOND_BEST_LR --gamma BEST_GAMMA
# check_experiment(10)

# 3. Save results
Zip the logs and models, then download /kaggle/working/Divine_results.zip from the Output panel before the session ends.

In [ ]:
!cd /kaggle/working/Deep-Q-Learning && zip -r /kaggle/working/Divine_results.zip logs/Divine models/Divine